# 05b — HCO và VNLS có phụ thuộc evaluator không?

Notebook 05 dùng HGB. Ở đây đổi sang **LightGBM nhỏ** với cùng hai cutoff, seed `41`, budget `8`, `alpha=0,85` và GroupKFold ba fold trong train. Câu hỏi không phải LightGBM tối ưu tham số, mà là ba thuật toán chọn feature có cùng thứ hạng khi thay evaluator không. Có đối chứng đầy đủ feature bằng **chính cấu hình LightGBM này**.

In [ ]:
from pathlib import Path
import sys
ROOT = next((candidate for candidate in (Path.cwd(), *Path.cwd().parents)
             if (candidate / 'research_code').is_dir() and (candidate / 'shared_code').is_dir()), None)
assert ROOT is not None, 'Open Jupyter from packages/ml or its notebooks directory.'
sys.path.insert(0, str(ROOT))
from shared_code.thuc_nghiem import (
    GOC, NGHIEN_CUU, DATA, KET_QUA, HINH, CAC_MOC,
    tao_thu_muc, duong_dan, chia_nhom, nap_du_lieu, chon_dac_trung,
    tien_xu_ly, bo_sung_ty_le, bo_sung_cua_so_vle,
)
tao_thu_muc()
print('Dữ liệu nguồn:', DATA)
print('Kết quả riêng:', KET_QUA)

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display
from lightgbm import LGBMClassifier
from research_code.models import ModelConfig, make_pipeline
from research_code.evaluation import metric_table
from research_code.optimizers import MaskObjective, run_search_budgeted

config = ModelConfig('LightGBM_search_80', LGBMClassifier(n_estimators=80, learning_rate=.055,
                     num_leaves=15, min_child_samples=35, verbosity=-1, random_state=41, n_jobs=4))
rows = []
for cutoff in (.5, .75):
    frame = nap_du_lieu(cutoff)
    features = chon_dac_trung(frame)
    train, valid = (frame[frame.split == value] for value in ('train','validation'))
    X_train, X_valid = train[features], valid[features]
    full_model = make_pipeline(X_train, config)
    full_model.fit(X_train, train.risk)
    full_metrics = metric_table(valid.risk, full_model.predict_proba(X_valid)[:,1])
    rows.append({'cutoff': cutoff, 'method': 'Full features', 'features': len(features),
                 'fitness_cv': np.nan, 'objective_calls': 0, **full_metrics})
    for method in ('HCO', 'VNLS', 'HCO-VNLS'):
        objective = MaskObjective(X_train, train.risk.to_numpy(), train.id_student.to_numpy(), config, alpha=.85)
        search = run_search_budgeted(method, objective, seed=41, budget=8, population=3)
        selected = search['selected_columns']
        model = make_pipeline(X_train[selected], config)
        model.fit(X_train[selected], train.risk)
        metrics = metric_table(valid.risk, model.predict_proba(X_valid[selected])[:,1])
        rows.append({'cutoff': cutoff, 'method': method, 'features': len(selected),
                     'fitness_cv': search['fitness_cv'], 'objective_calls': search['objective_calls'],
                     'selected_columns': '|'.join(selected), **metrics})
result = pd.DataFrame(rows)
result.to_csv(KET_QUA/'hco_vnls_lightgbm_validation.csv', index=False, encoding='utf-8-sig')
display(result[['cutoff','method','features','accuracy','recall_risk','f1_risk',
                'average_precision_risk','fitness_cv','objective_calls']].round(4))

## Cách kết luận

So sánh HCO, VNLS và HCO–VNLS **trong cùng cutoff** với hàng `Full features`. Nếu HCO–VNLS tăng fitness nhưng không tăng AP thì lợi ích nằm ở giảm số feature. Đây chỉ một seed; không được kết luận model nào hay phương pháp nào luôn hơn. `LightGBM_search_80` khác cấu hình 210 cây ở notebook 04, nên không trộn hai bảng để quy tác động riêng cho feature selection.

## Kết quả đọc nhanh — lần chạy 23/09/2026

Ở 75%, LightGBM 80 cây đủ `38` feature đạt Accuracy `0,8735`, Recall `0,6898`, AP `0,8970`; HCO–VNLS giữ `17` feature đạt Accuracy `0,8741`, Recall `0,6898`, AP `0,8968`. Lợi ích chính là giảm 21 feature mà gần như giữ AP. Ở 50%, hybrid giảm cả Accuracy, Recall, AP so với đủ feature. Chỉ một seed: kết quả cần lặp lại trước khi kết luận.